In [2]:
# =========================
# RUN SAVER (paste into ALL notebooks)
# =========================
from pathlib import Path
import json, shutil, datetime, platform, sys
import numpy as np
import pandas as pd

# Change this if you want a different location (this is safe + simple on Windows)
RUNS_ROOT = Path.home() / "rr_runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def _ts():
    return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

def make_run_dir(method: str, dataset: str, tag: str = "") -> Path:
    tag = f"_{tag}" if tag else ""
    run_dir = RUNS_ROOT / method / dataset / f"{_ts()}{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def env_info():
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }

def save_run(run_dir: Path,
             per_window_df: pd.DataFrame,
             per_subject_df: pd.DataFrame,
             config: dict,
             splits_path: str | None = None):
    per_window_path = run_dir / "per_window.csv"
    per_subject_path = run_dir / "per_subject.csv"
    config_path = run_dir / "config.json"

    per_window_df.to_csv(per_window_path, index=False)
    per_subject_df.to_csv(per_subject_path, index=False)

    config = dict(config)
    config["environment"] = env_info()
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    if splits_path is not None:
        sp = Path(splits_path)
        if sp.exists():
            shutil.copy2(sp, run_dir / sp.name)

    print("\n✅ Saved run to:", str(run_dir))
    print("  -", per_window_path.name, "| rows:", len(per_window_df))
    print("  -", per_subject_path.name, "| rows:", len(per_subject_df))
    print("  -", config_path.name)
    if splits_path is not None:
        print("  - splits:", Path(splits_path).name, ("OK" if Path(splits_path).exists() else "MISSING"))


In [3]:
import os, json, gc, pickle, warnings, math, random
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import FastICA
from sklearn.exceptions import ConvergenceWarning

from scipy.signal import welch

def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [4]:
# ====== EDIT THESE PATHS ======
PPG_DALIA_RAW_PKL = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Raw_Signal.pkl"
PPG_DALIA_ANN_PKL = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Annotation.pkl"
WESAD_RAW_PKL     = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Raw_Signal.pkl"
WESAD_ANN_PKL     = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Annotation.pkl"

# where your saved splits actually are (baseline step)

import os, json, shutil

OUT_DIR = r"C:\Users\yasmi\OneDrive\Документы\kazemi_fusion_out"
os.makedirs(OUT_DIR, exist_ok=True)

BASELINE_OUT_DIR = r"C:\Jupyter Files\baseline_outputs"
SPLITS_JSON = os.path.join(BASELINE_OUT_DIR, "splits_loso.json")


print("SPLITS_JSON:", SPLITS_JSON)
print("exists?:", os.path.exists(SPLITS_JSON))
assert os.path.exists(SPLITS_JSON), "Splits JSON path is wrong"
with open(SPLITS_JSON, "r", encoding="utf-8") as f:
    splits = json.load(f)

print("Loaded splits keys:", list(splits.keys()))

# ====== Fixed window assumptions (your assets already satisfy this) ======
FS = 64.0
WIN_SAMPLES = 2048

# ====== Channel mapping (based on your sweep) ======
# PPG channel indices:
WESAD_PPG_CH = 1
DALIA_PPG_CH = 0

# IMU channels:
# Assumption: remaining channels after PPG are [ACCx, ACCy, ACCz, GYRx, GYRy, GYRz] if present.
# Your WESAD raw has 4 channels total -> likely PPG + 3-axis ACC (no gyro).
WESAD_ACC_CHS = [0, 2, 3]
WESAD_GYR_CHS = None  # no gyro

# Your DaLiA raw has 7 channels total -> likely PPG + 3-axis ACC + 3-axis GYR.
DALIA_ACC_CHS = [1, 2, 3]
DALIA_GYR_CHS = [4, 5, 6]


SPLITS_JSON: C:\Jupyter Files\baseline_outputs\splits_loso.json
exists?: True
Loaded splits keys: ['protocol', 'WESAD_subjects', 'PPG_DaLiA_subjects', 'folds_WESAD', 'folds_PPG_DaLiA']


In [5]:
def load_pkl(path):
    assert os.path.exists(path), f"File not found: {path}"
    with open(path, "rb") as f:
        return pickle.load(f)

def load_kazemi_raw(pkl_path: str) -> np.ndarray:
    obj = load_pkl(pkl_path)
    X = np.asarray(obj)
    return X

def load_kazemi_annotation_df(pkl_path: str) -> pd.DataFrame:
    obj = load_pkl(pkl_path)
    assert isinstance(obj, pd.DataFrame), f"Expected DataFrame, got {type(obj)}"
    return obj

def get_rr_sid_from_ann(df: pd.DataFrame):
    # expected columns: ['Reference_RR','activity_id','patient_id']
    rr = df["Reference_RR"].to_numpy(dtype=np.float32)
    sid = df["patient_id"].to_numpy(dtype=int)
    return rr, sid

def windowing_checks(X, rr, sid, name):
    X = np.asarray(X)
    print(f"\n==== {name} checks ====")
    print("X:", X.shape, "rr:", rr.shape, "sid:", sid.shape)
    assert X.ndim == 3, "Expected X as (N,C,L)"
    assert X.shape[0] == rr.shape[0] == sid.shape[0], "N mismatch"
    assert X.shape[2] == WIN_SAMPLES, f"Expected L={WIN_SAMPLES}"
    print("subjects:", sorted(np.unique(sid).tolist()))


In [6]:
wesad_raw = load_kazemi_raw(WESAD_RAW_PKL)
dalia_raw = load_kazemi_raw(PPG_DALIA_RAW_PKL)

wesad_ann_df = load_kazemi_annotation_df(WESAD_ANN_PKL)
dalia_ann_df = load_kazemi_annotation_df(PPG_DALIA_ANN_PKL)

wesad_rr, wesad_sid = get_rr_sid_from_ann(wesad_ann_df)
dalia_rr, dalia_sid = get_rr_sid_from_ann(dalia_ann_df)

windowing_checks(wesad_raw, wesad_rr, wesad_sid, "WESAD")
windowing_checks(dalia_raw, dalia_rr, dalia_sid, "PPG-DaLiA")



==== WESAD checks ====
X: (1797, 4, 2048) rr: (1797,) sid: (1797,)
subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

==== PPG-DaLiA checks ====
X: (3883, 7, 2048) rr: (3883,) sid: (3883,)
subjects: [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [7]:
def minmax_to_minus1_1(x: np.ndarray, eps=1e-8) -> np.ndarray:
    x = np.asarray(x)
    mn = np.min(x)
    mx = np.max(x)
    if (mx - mn) < eps:
        return np.zeros_like(x, dtype=np.float32)
    y = (x - mn) / (mx - mn)
    y = y * 2.0 - 1.0
    return y.astype(np.float32)

def pick_ica_resp(sig_3axis: np.ndarray) -> np.ndarray:
    """
    sig_3axis: (3,L) -> returns (L,)
    If ICA fails, uses SVD first component.
    """
    X = sig_3axis.T.astype(np.float64)  # (L,3)
    X = X - X.mean(axis=0, keepdims=True)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        try:
            ica = FastICA(
                n_components=3,
                random_state=0,
                max_iter=1000,
                tol=1e-3,
                whiten="unit-variance"
            )
            comps = ica.fit_transform(X)  # (L,3)
        except Exception:
            comps = None

    if comps is None or (not np.isfinite(comps).all()):
        # fallback SVD
        U, S, Vt = np.linalg.svd(X, full_matrices=False)
        comp = U[:, 0]
        comp = comp - comp.mean()
        comp = comp / (comp.std() + 1e-8)
        return comp.astype(np.float32)

    # choose component with strongest energy in respiration band
    # respiration band (Hz): ~0.1 - 0.7 (6-42 bpm)
    f, P = welch(comps, fs=FS, nperseg=256, axis=0)
    band = (f >= 0.10) & (f <= 0.70)
    band_power = P[band, :].sum(axis=0)
    k = int(np.argmax(band_power))
    comp = comps[:, k]
    comp = comp - comp.mean()
    comp = comp / (comp.std() + 1e-8)
    return comp.astype(np.float32)

def build_fusion_inputs(raw: np.ndarray, ppg_ch: int, acc_chs, gyr_chs) -> np.ndarray:
    """
    raw: (N,C,L)
    returns: fusion_X (N,3,L) = [PPG, Resp(ACC), Resp(GYR)]
    if gyro missing -> Resp(GYR)=Resp(ACC)
    """
    raw = np.asarray(raw)
    assert raw.ndim == 3, f"Expected (N,C,L), got {raw.shape}"
    N, C, L = raw.shape

    ppg = raw[:, ppg_ch, :].astype(np.float32)
    acc = raw[:, acc_chs, :].astype(np.float32)  # (N,3,L)
    gyr = None if gyr_chs is None else raw[:, gyr_chs, :].astype(np.float32)

    fusion = np.zeros((N, 3, L), dtype=np.float32)

    for i in tqdm(range(N), desc="ICA extract"):
        resp_acc = pick_ica_resp(acc[i])
        if gyr is None:
            resp_gyr = resp_acc
        else:
            resp_gyr = pick_ica_resp(gyr[i])

        fusion[i, 0, :] = minmax_to_minus1_1(ppg[i])
        fusion[i, 1, :] = minmax_to_minus1_1(resp_acc)
        fusion[i, 2, :] = minmax_to_minus1_1(resp_gyr)

    return fusion

wesad_X = build_fusion_inputs(wesad_raw, WESAD_PPG_CH, WESAD_ACC_CHS, WESAD_GYR_CHS)
dalia_X = build_fusion_inputs(dalia_raw, DALIA_PPG_CH, DALIA_ACC_CHS, DALIA_GYR_CHS)

print("WESAD fusion:", wesad_X.shape)
print("DaLiA fusion:", dalia_X.shape)

del wesad_raw, dalia_raw
gc.collect()


ICA extract: 100%|██████████| 3883/3883 [02:07<00:00, 30.56it/s]

WESAD fusion: (1797, 3, 2048)
DaLiA fusion: (3883, 3, 2048)


9

In [8]:
from scipy.signal import butter, filtfilt, welch
import numpy as np

def bandpass(x, fs, lo=0.1, hi=0.7, order=3):
    ny = 0.5 * fs
    b, a = butter(order, [lo/ny, hi/ny], btype="band")
    return filtfilt(b, a, x)

def compute_ppg_sqi01(ppg_1d: np.ndarray, fs: float) -> float:
    x = np.asarray(ppg_1d, dtype=np.float64)
    if not np.isfinite(x).all():
        return 0.0

    x = x - np.mean(x)
    s = np.std(x)
    if s < 1e-6:
        return 0.0
    x = x / (s + 1e-12)

    # time-domain resp-band energy ratio
    try:
        xb = bandpass(x, fs, 0.10, 0.70)
    except Exception:
        return 0.0

    v_band = float(np.var(xb))
    v_tot = float(np.var(x)) + 1e-12
    ratio_td = v_band / v_tot

    # frequency-domain band power ratio
    # IMPORTANT: use larger nperseg for better resolution in 0.1–0.7 Hz band
    nperseg = min(1024, len(x))  # 1024 -> Δf = 0.0625 Hz at fs=64
    f, P = welch(x, fs=fs, nperseg=nperseg)

    band = (f >= 0.10) & (f <= 0.70)
    tot  = (f >= 0.05) & (f <= 2.0)

    # with nperseg=1024, band will have plenty of bins; but allow >=2 to be safe
    if band.sum() < 2 or tot.sum() < 2:
        return 0.0

    bp = float(np.sum(P[band]))
    tp = float(np.sum(P[tot])) + 1e-12
    ratio_fd = bp / tp

    raw = 0.7 * ratio_td + 0.3 * ratio_fd
    sqi01 = raw / (raw + 0.15)  # squash into 0..1

    if not np.isfinite(sqi01):
        return 0.0
    return float(np.clip(sqi01, 0.0, 1.0))

def compute_sqi_array01(ppg_windows: np.ndarray, fs: float) -> np.ndarray:
    N = ppg_windows.shape[0]
    out = np.zeros(N, dtype=np.float32)
    for i in tqdm(range(N), desc="Compute SQI01"):
        out[i] = compute_ppg_sqi01(ppg_windows[i], fs)
    return out


In [9]:
wesad_sqi_raw = compute_sqi_array01(wesad_X[:, 0, :], FS)
dalia_sqi_raw = compute_sqi_array01(dalia_X[:, 0, :], FS)

def sqi_stats(name, s):
    s = np.asarray(s)
    print(name, "finite", np.isfinite(s).mean(),
          "min", float(np.min(s)),
          "median", float(np.median(s)),
          "max", float(np.max(s)))

sqi_stats("WESAD SQI01", wesad_sqi_raw)
sqi_stats("DaLiA SQI01", dalia_sqi_raw)


Compute SQI01: 100%|██████████| 3883/3883 [00:07<00:00, 487.64it/s]


WESAD SQI01 finite 1.0 min 0.17975342273712158 median 0.6794701218605042 max 0.8593628406524658
DaLiA SQI01 finite 1.0 min 0.00672891503199935 median 0.3266887664794922 max 0.8206043839454651


In [10]:
def compute_motion_from_fusion(X_fusion):
    """
    X_fusion: (N,3,L) where channels are [PPG, resp_acc, resp_gyr]
    Motion proxy = sqrt(mean(resp_acc^2) + mean(resp_gyr^2))
    """
    X_fusion = np.asarray(X_fusion, dtype=np.float32)
    a = X_fusion[:, 1, :]
    g = X_fusion[:, 2, :]
    return np.sqrt(np.mean(a * a, axis=1) + np.mean(g * g, axis=1)).astype(np.float32)

wesad_motion = compute_motion_from_fusion(wesad_X)
dalia_motion = compute_motion_from_fusion(dalia_X)

# convenience aliases (avoid name mismatches later)
wesad_X_fusion = wesad_X
dalia_X_fusion = dalia_X
wesad_sqi01 = wesad_sqi_raw
dalia_sqi01 = dalia_sqi_raw

print("WESAD motion:", wesad_motion.shape, "range", float(wesad_motion.min()), float(wesad_motion.max()))
print("DaLiA motion:", dalia_motion.shape, "range", float(dalia_motion.min()), float(dalia_motion.max()))


WESAD motion: (1797,) range 0.16644287109375 1.146083116531372
DaLiA motion: (3883,) range 0.1427905559539795 1.005242943763733


In [11]:
# =========================
# Deterministic LOSO fold iterator + deterministic VAL selection
# =========================
import numpy as np
import pandas as pd
import time

def iter_folds_from_splits_anyformat(splits: dict, dataset_name: str):
    """
    Supports your splits keys:
      - folds_WESAD
      - folds_PPG_DaLiA
    And fold dict formats:
      - {"train_subjects":[...], "test_subjects":[...]}
      - {"train":[...], "test":[...]}
    """
    if dataset_name.lower().startswith("wesad"):
        key = "folds_WESAD"
    else:
        key = "folds_PPG_DaLiA"

    if key not in splits:
        raise KeyError(f"Missing '{key}' in splits. Available keys: {list(splits.keys())}")

    folds_obj = splits[key]
    if not isinstance(folds_obj, list) or len(folds_obj) == 0:
        raise ValueError(f"Bad splits format: splits['{key}'] must be a non-empty list")

    for fold in folds_obj:
        if "train_subjects" in fold and "test_subjects" in fold:
            train = [int(s) for s in fold["train_subjects"]]
            test  = [int(s) for s in fold["test_subjects"]]
        elif "train" in fold and "test" in fold:
            train = [int(s) for s in fold["train"]]
            test  = [int(s) for s in fold["test"]]
        else:
            raise ValueError(f"Bad fold keys: {fold.keys()}")
        yield {"train": train, "test": test}

def pick_val_subject_deterministic(train_subjects):
    """Deterministic validation subject: smallest subject id in training."""
    train_subjects = [int(s) for s in train_subjects]
    if len(train_subjects) < 2:
        raise ValueError("Need at least 2 training subjects to pick a val subject.")
    return int(np.min(train_subjects))


In [12]:
class WindowDatasetFeat(Dataset):
    def __init__(self, X, y, sid, sqi01, motion, keep_subjects, seed=0, max_windows=None):
        keep = np.asarray(list(keep_subjects), dtype=int)
        idx = np.where(np.isin(sid, keep))[0]
        if max_windows is not None and len(idx) > max_windows:
            rng = np.random.default_rng(seed)
            idx = rng.choice(idx, size=max_windows, replace=False)
        self.idx = idx.astype(int)
        self.X = X
        self.y = y
        self.sid = sid
        self.sqi01 = sqi01
        self.motion = motion

        m = self.motion[self.idx]
        self.m_mu = float(np.mean(m))
        self.m_sd = float(np.std(m) + 1e-6)

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        xb = torch.from_numpy(self.X[j].astype(np.float32))
        yb = torch.tensor(np.float32(self.y[j]))
        sq = float(self.sqi01[j])
        mo = float((self.motion[j] - self.m_mu) / self.m_sd)
        feat = torch.tensor([sq, mo], dtype=torch.float32)
        return xb, yb, feat, torch.tensor(int(self.sid[j])), torch.tensor(int(j))


In [13]:
import torch
import torch.nn as nn

class ResidualInceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, slope=0.2):
        super().__init__()
        b = out_ch // 3
        b1, b2, b3 = b, b, out_ch - 2*b

        self.br1 = nn.Conv1d(in_ch, b1, 3, stride=2, padding=1, dilation=1, bias=False)
        self.br2 = nn.Conv1d(in_ch, b2, 3, stride=2, padding=2, dilation=2, bias=False)
        self.br3 = nn.Conv1d(in_ch, b3, 3, stride=2, padding=4, dilation=4, bias=False)

        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.LeakyReLU(slope, inplace=True)
        self.res = nn.Conv1d(in_ch, out_ch, 1, stride=2, padding=0, bias=False)

    def forward(self, x):
        y = torch.cat([self.br1(x), self.br2(x), self.br3(x)], dim=1)
        y = self.act(self.bn(y))
        return y + self.res(x)

class KazemiRRNet(nn.Module):
    def __init__(self, in_ch=3, slope=0.2):
        super().__init__()
        filters = [8, 16, 32, 64, 128, 256, 512, 1024]
        blocks = []
        ch = in_ch
        for f in filters:
            blocks.append(ResidualInceptionBlock(ch, f, slope=slope))
            ch = f
        self.backbone = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(ch, 64)
        self.fc2 = nn.Linear(64, 1)
        self.act = nn.LeakyReLU(slope, inplace=True)

    def forward(self, x):
        z = self.backbone(x)
        z = self.gap(z).squeeze(-1)
        z = self.act(self.fc1(z))
        return self.fc2(z).squeeze(-1)


In [14]:
class KazemiWithResidual(nn.Module):
    def __init__(self, in_ch=3, feat_dim=2, hidden=32):
        super().__init__()
        self.base = KazemiRRNet(in_ch=in_ch)
        self.res = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, xb, feat):
        # xb: (B,3,L), feat: (B,2)
        y_base = self.base(xb)                      # (B,)
        delta = self.res(feat).squeeze(-1)          # (B,)
        return y_base + delta, y_base, delta


In [1]:
import time
from torch.utils.data import RandomSampler

def train_one_fold_v6(
    X, y, sid, sqi01, motion,
    train_subs, val_sub, test_subs,
    device,
    epochs=60,
    steps_per_epoch=120,
    batch_size=64,
    lr=5e-4,
    weight_decay=1e-4,
    patience=10,
    seed=123,
    delta_l2=0.01,
):
    seed_everything(seed)

    ds_train = WindowDatasetFeat(X, y, sid, sqi01, motion, keep_subjects=train_subs, seed=seed)
    ds_val   = WindowDatasetFeat(X, y, sid, sqi01, motion, keep_subjects=[val_sub], seed=seed)
    ds_test  = WindowDatasetFeat(X, y, sid, sqi01, motion, keep_subjects=test_subs, seed=seed)

    # Match “strong” style: fixed number of gradient steps per epoch via replacement sampling
    sampler = RandomSampler(ds_train, replacement=True, num_samples=steps_per_epoch * batch_size)
    dl_train = DataLoader(ds_train, batch_size=batch_size, sampler=sampler, drop_last=True)

    dl_val  = DataLoader(ds_val,  batch_size=256, shuffle=False)
    dl_test = DataLoader(ds_test, batch_size=256, shuffle=False)

    model = KazemiWithResidual(in_ch=X.shape[1], feat_dim=2, hidden=32).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)



In [16]:
def eval_strat_and_coverage_from_df(df, motion_all, sqi_all, dataset_name,
                                   cover_thresholds=(0.10,0.20,0.30,0.40,0.50)):
    df = df.copy()
    idx = df["window_index"].astype(int).to_numpy()

    df["motion"] = motion_all[idx]
    df["sqi01"] = sqi_all[idx]
    df["abs_err"] = np.abs(df["rr_pred"].to_numpy() - df["rr_ref"].to_numpy())

    q1, q2 = np.quantile(df["motion"].to_numpy(), [1/3, 2/3])
    df["motion_bin"] = np.where(df["motion"] <= q1, "low",
                         np.where(df["motion"] <= q2, "mid", "high"))

    rows = []
    for b in ["all", "low", "mid", "high"]:
        d = df if b == "all" else df[df["motion_bin"] == b]
        mae = float(d["abs_err"].mean())
        rmse = float(np.sqrt(np.mean((d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy())**2)))
        within3 = float(np.mean(d["abs_err"].to_numpy() <= 3.0))
        rows.append([dataset_name, b, int(len(d)), mae, rmse, within3])

    strat = pd.DataFrame(rows, columns=["dataset","motion_bin","N","MAE","RMSE","pct_|err|<=3"])

    cov_rows = []
    for t in cover_thresholds:
        keep = df["sqi01"].to_numpy() >= float(t)
        cov = float(np.mean(keep))
        if keep.sum() == 0:
            cov_rows.append([dataset_name, t, cov, np.nan, np.nan, np.nan])
        else:
            d = df[keep]
            mae = float(d["abs_err"].mean())
            rmse = float(np.sqrt(np.mean((d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy())**2)))
            within3 = float(np.mean(d["abs_err"].to_numpy() <= 3.0))
            cov_rows.append([dataset_name, t, cov, mae, rmse, within3])

    cov = pd.DataFrame(cov_rows, columns=["dataset","sqi_threshold","coverage","MAE","RMSE","pct_|err|<=3"])
    return strat, cov

print("Eval helper ready (no df_test required).")


Eval helper ready (no df_test required).


In [19]:
import os, time
import pandas as pd
import numpy as np

# ----------------------------
# Checkpoint helpers
# ----------------------------
def _safe_read_csv(path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            return pd.read_csv(path)
        except Exception:
            return None
    return None

def _append_and_save_csv(path, df_new):
    if df_new is None or len(df_new) == 0:
        return
    df_old = _safe_read_csv(path)
    if df_old is None:
        df_new.to_csv(path, index=False)
    else:
        out = pd.concat([df_old, df_new], ignore_index=True)
        out.to_csv(path, index=False)

# ----------------------------
# MAIN: LOSO with fold checkpoint saving
# ----------------------------
def run_loso_proposed_v6_checkpoint(
    dataset_name,
    X, y, sid,
    sqi01, motion,
    splits,
    device,
    epochs=60,
    steps_per_epoch=120,
    batch_size=64,
    lr=5e-4,
    weight_decay=1e-4,
    patience=10,
    seed=123,
    method_name="proposed_v6_tuned",
    tag="ckpt",
    splits_path=None,
):
    # create ONE run folder for the whole dataset
    run_dir = make_run_dir(method_name, dataset_name, tag=tag)
    print("\nRUN DIR:", run_dir)

    # checkpoint file paths
    pw_path = os.path.join(run_dir, "per_window_partial.csv")
    ps_path = os.path.join(run_dir, "per_subject_partial.csv")

    folds = list(iter_folds_from_splits_anyformat(splits, dataset_name))
    if len(folds) == 0:
        raise ValueError(f"No folds found for {dataset_name}")

    # If resuming, determine which folds already saved
    done_folds = set()
    ps_existing = _safe_read_csv(ps_path)
    if ps_existing is not None and "fold" in ps_existing.columns:
        done_folds = set(ps_existing["fold"].astype(int).tolist())
        print("Resume: already have folds:", sorted(done_folds))

    for fold_i, fold in enumerate(folds, start=1):
        if fold_i in done_folds:
            print(f"[SKIP] {dataset_name} fold {fold_i}/{len(folds)} already saved.")
            continue

        test_sub = int(fold["test"][0])
        train_full = [int(s) for s in fold["train"]]

        val_sub = pick_val_subject_deterministic(train_full)
        train_subs = [s for s in train_full if s != val_sub]
        test_subs = [test_sub]

        print(f"\n[PROPOSED v6 TUNED] {dataset_name} fold {fold_i}/{len(folds)} | test={test_sub} | val={val_sub} | train_n={len(train_subs)}")
        print(f"settings: epochs={epochs}, steps/epoch={steps_per_epoch}, batch={batch_size}, lr={lr}, wd={weight_decay}, patience={patience}")

        t0 = time.time()
        df_fold, met = train_one_fold_v6(
            X, y, sid, sqi01, motion,
            train_subs=train_subs,
            val_sub=val_sub,
            test_subs=test_subs,
            device=device,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            patience=patience,
            seed=seed + fold_i
        )
        sec = time.time() - t0

        # enforce fold metadata columns
        df_fold = df_fold.copy()
        df_fold["dataset"] = dataset_name
        df_fold["fold"] = int(fold_i)
        df_fold["subject_id"] = int(test_sub)
        df_fold["val_subject"] = int(val_sub)

        met = dict(met)
        met.update({
            "dataset": dataset_name,
            "fold": int(fold_i),
            "subject_id": int(test_sub),
            "val_subject": int(val_sub),
            "seconds": float(sec)
        })
        df_met = pd.DataFrame([met])

        # checkpoint save
        _append_and_save_csv(pw_path, df_fold)
        _append_and_save_csv(ps_path, df_met)

        print(f"[fold {fold_i}] saved checkpoint | seconds={sec:.1f}")

    # finalize: read partials and save as official per_window/per_subject using your saver
    df_all = _safe_read_csv(pw_path)
    df_sub = _safe_read_csv(ps_path)
    if df_all is None or df_sub is None:
        raise RuntimeError("No checkpoint outputs found to finalize.")

    config = dict(
        method=method_name,
        tag=tag,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        batch_size=batch_size,
        lr=lr,
        weight_decay=weight_decay,
        patience=patience,
        seed=seed,
        note="Checkpoint saving after every fold; deterministic val subject"
    )

    save_run(run_dir, df_all, df_sub, config, splits_path=splits_path)
    return run_dir, df_all, df_sub


In [22]:
EPOCHS = 60
STEPS_PER_EPOCH = 120
BATCH = 64
LR = 5e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
SEED = 123

# WESAD
run_dir_w, wesad_all, wesad_sub = run_loso_proposed_v6_checkpoint(
    "WESAD",
    wesad_X, wesad_rr, wesad_sid,
    sqi01=wesad_sqi_raw,
    motion=wesad_motion,
    splits=splits,
    device=device,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    method_name="proposed_v6_tuned",
    tag="ckpt_detval",
    splits_path=SPLITS_JSON if "SPLITS_JSON" in globals() else None
)

# DaLiA
run_dir_d, dalia_all, dalia_sub = run_loso_proposed_v6_checkpoint(
    "PPG-DaLiA",
    dalia_X, dalia_rr, dalia_sid,
    sqi01=dalia_sqi_raw,
    motion=dalia_motion,
    splits=splits,
    device=device,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=BATCH,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    method_name="proposed_v6_tuned",
    tag="ckpt_detval",
    splits_path=SPLITS_JSON if "SPLITS_JSON" in globals() else None
)



RUN DIR: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_133616_ckpt_detval

[PROPOSED v6 TUNED] WESAD fold 1/10 | test=2 | val=3 | train_n=8
settings: epochs=60, steps/epoch=120, batch=64, lr=0.0005, wd=0.0001, patience=10
  epoch 1/60 | train_loss=3.2487 | val_mae=3.124 (val_subject=3)
  epoch 2/60 | train_loss=1.6702 | val_mae=3.053 (val_subject=3)


KeyboardInterrupt: 

In [23]:
from pathlib import Path
runs_root = Path(r"C:\Users\yasmi\rr_runs")  # change if needed

method = "proposed_v6_tuned"
dataset = "WESAD"

p = runs_root / method / dataset
print("Looking in:", p)
if p.exists():
    for d in sorted(p.glob("*"), reverse=True)[:10]:
        print(d)
        # show what files exist inside
        for f in d.glob("*"):
            print("   ", f.name)
else:
    print("Folder not found.")


Looking in: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_133616_ckpt_detval
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval
    per_subject_partial.csv
    per_window_partial.csv
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_105330_ckpt_detval
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260127_133744
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260127_124555
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260127_124526
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_201212
    config.json
    per_subject.csv
    per_window.csv
    splits_loso.json
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_195354
C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260126_195331


In [29]:
# =========================
# RESUME: Proposed v6 tuned (det-val) — DaLiA from per_*_partial.csv
# =========================
import os, time
import pandas as pd
import numpy as np

RUN_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval"

# make sure the folder exists
os.makedirs(RUN_DIR, exist_ok=True)

PER_W_PART = os.path.join(RUN_DIR, "per_window_partial.csv")
PER_S_PART = os.path.join(RUN_DIR, "per_subject_partial.csv")

def _safe_write_csv(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

# ---- sanity: required vars must exist in memory ----
required = ["splits", "device", "iter_folds_from_splits_anyformat", "pick_val_subject_deterministic",
            "train_one_fold_v6", "dalia_rr", "dalia_sid"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"Kernel is missing required variables/functions: {missing}\n"
                    f"Re-run your setup/data/model cells first, then run this resume cell again.")

# Choose the input array your proposed method expects:
if "dalia_X_fusion" in globals():
    X_USE = dalia_X_fusion
elif "dalia_X" in globals():
    X_USE = dalia_X
else:
    raise NameError("Need dalia_X_fusion or dalia_X in memory (re-run your fusion build cell).")

# SQI + motion
if "dalia_sqi01" in globals():
    SQI = dalia_sqi01
elif "dalia_sqi_raw" in globals():
    SQI = dalia_sqi_raw
else:
    raise NameError("Need dalia_sqi01 or dalia_sqi_raw in memory.")

if "dalia_motion" not in globals():
    raise NameError("Need dalia_motion in memory.")

# ---- settings (match your run) ----
EPOCHS = 60
STEPS_PER_EPOCH = 120
BATCH = 64
LR = 5e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
SEED = 123

# ---- load folds ----
folds = list(iter_folds_from_splits_anyformat(splits, "PPG-DaLiA"))
n_folds = len(folds)
print("Found folds:", n_folds)

# ---- load partials if present ----
if os.path.exists(PER_W_PART):
    df_all = pd.read_csv(PER_W_PART)
else:
    df_all = pd.DataFrame()

if os.path.exists(PER_S_PART):
    df_sub = pd.read_csv(PER_S_PART)
else:
    df_sub = pd.DataFrame()

done_folds = set()
if len(df_all) and "fold" in df_all.columns:
    done_folds |= set(df_all["fold"].dropna().astype(int).unique().tolist())
if len(df_sub) and "fold" in df_sub.columns:
    done_folds |= set(df_sub["fold"].dropna().astype(int).unique().tolist())

print("Already completed folds (from partials):", sorted(done_folds))

# ---- resume loop ----
for fold_i, fold in enumerate(folds, start=1):
    if fold_i in done_folds:
        print(f"[SKIP] fold {fold_i}/{n_folds} already saved")
        continue

    test_sub = int(fold["test"][0])
    train_full = [int(s) for s in fold["train"]]

    val_sub = pick_val_subject_deterministic(train_full)
    train_subs = [s for s in train_full if s != val_sub]
    test_subs = [test_sub]

    print(f"\n[RESUME PROPOSED v6] fold {fold_i}/{n_folds} | test={test_sub} | val={val_sub} | train_n={len(train_subs)}")
    t0 = time.time()

    # ---- run 1 fold ----
    # IMPORTANT: train_one_fold_v6 must return: (df_test, metrics_dict)
    df_fold, met = train_one_fold_v6(
        X_USE, dalia_rr, dalia_sid, SQI, dalia_motion,
        train_subs=train_subs,
        val_sub=val_sub,
        test_subs=test_subs,
        device=device,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        batch_size=BATCH,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        seed=SEED + fold_i
    )

    sec = time.time() - t0
    met = dict(met)
    met.update({"dataset": "PPG-DaLiA", "fold": int(fold_i), "subject_id": int(test_sub), "val_subject": int(val_sub), "seconds": float(sec)})

    # ---- stamp df_fold ----
    df_fold = df_fold.copy()
    df_fold["dataset"] = "PPG-DaLiA"
    df_fold["fold"] = int(fold_i)
    df_fold["subject_id"] = int(test_sub)
    df_fold["val_subject"] = int(val_sub)

    # ---- append + save partials immediately ----
    df_all = pd.concat([df_all, df_fold], ignore_index=True) if len(df_all) else df_fold
    df_sub = pd.concat([df_sub, pd.DataFrame([met])], ignore_index=True) if len(df_sub) else pd.DataFrame([met])

    _safe_write_csv(df_all, PER_W_PART)
    _safe_write_csv(df_sub, PER_S_PART)

    print(f"[fold {fold_i}] saved partials | per_window rows={len(df_all)} | per_subject rows={len(df_sub)}")

print("\n✅ Resume loop finished.")
print("Partial files:")
print(" -", PER_W_PART, "| exists:", os.path.exists(PER_W_PART))
print(" -", PER_S_PART, "| exists:", os.path.exists(PER_S_PART))


Found folds: 14
Already completed folds (from partials): [1, 2, 3, 4, 5, 6, 7, 8]
[SKIP] fold 1/14 already saved
[SKIP] fold 2/14 already saved
[SKIP] fold 3/14 already saved
[SKIP] fold 4/14 already saved
[SKIP] fold 5/14 already saved
[SKIP] fold 6/14 already saved
[SKIP] fold 7/14 already saved
[SKIP] fold 8/14 already saved

[RESUME PROPOSED v6] fold 9/14 | test=10 | val=1 | train_n=12
  epoch 1/60 | train_loss=3.7167 | val_mae=2.886 (val_subject=1)
  epoch 2/60 | train_loss=2.2412 | val_mae=3.493 (val_subject=1)
  epoch 3/60 | train_loss=1.8458 | val_mae=3.035 (val_subject=1)
  epoch 4/60 | train_loss=1.4526 | val_mae=3.084 (val_subject=1)
  epoch 5/60 | train_loss=1.3343 | val_mae=3.177 (val_subject=1)
  epoch 6/60 | train_loss=1.0162 | val_mae=3.372 (val_subject=1)
  epoch 7/60 | train_loss=0.9431 | val_mae=3.148 (val_subject=1)
  epoch 8/60 | train_loss=0.7932 | val_mae=3.207 (val_subject=1)
  epoch 9/60 | train_loss=0.7821 | val_mae=3.174 (val_subject=1)
  epoch 10/60 | train_

In [30]:
import sys, platform, numpy as np, pandas as pd
try:
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torchv = torch.__version__
except Exception:
    dev = "unknown"
    torchv = "unknown"

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torchv)
print("device:", dev)


python: 3.13.5
platform: Windows-11-10.0.26100-SP0
numpy: 2.1.3
pandas: 2.2.3
torch: 2.9.1+cpu
device: cpu


In [31]:
import json, os, numpy as np

print("has splits in memory:", "splits" in globals())
if "SPLITS_JSON" in globals():
    print("SPLITS_JSON:", SPLITS_JSON, "| exists:", os.path.exists(SPLITS_JSON))

# show splits keys if present
if "splits" in globals():
    print("splits keys:", list(splits.keys()))

# quick fold peek (works with your anyformat iterator)
def _peek(dataset):
    folds = list(iter_folds_from_splits_anyformat(splits, dataset))
    print("\n===", dataset, "===")
    print("n_folds:", len(folds))
    print("fold1 keys:", folds[0].keys())
    print("fold1 train:", folds[0]["train"])
    print("fold1 test :", folds[0]["test"])
    return folds

_ = _peek("WESAD")
_ = _peek("PPG-DaLiA")


has splits in memory: True
SPLITS_JSON: C:\Jupyter Files\baseline_outputs\splits_loso.json | exists: True
splits keys: ['protocol', 'WESAD_subjects', 'PPG_DaLiA_subjects', 'folds_WESAD', 'folds_PPG_DaLiA']

=== WESAD ===
n_folds: 10
fold1 keys: dict_keys(['train', 'test'])
fold1 train: [3, 4, 5, 6, 7, 8, 9, 10, 11]
fold1 test : [2]

=== PPG-DaLiA ===
n_folds: 14
fold1 keys: dict_keys(['train', 'test'])
fold1 train: [2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]
fold1 test : [1]


In [33]:
import os, json, shutil
import pandas as pd

SPLITS_JSON = r"C:\Jupyter Files\baseline_outputs\splits_loso.json"
assert os.path.exists(SPLITS_JSON), f"Missing SPLITS_JSON: {SPLITS_JSON}"

PROPOSED_WESAD_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval"
PROPOSED_DALIA_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval"

def _safe_write_csv(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def finalize_partials(run_dir):
    pw_part = os.path.join(run_dir, "per_window_partial.csv")
    ps_part = os.path.join(run_dir, "per_subject_partial.csv")
    pw_out  = os.path.join(run_dir, "per_window.csv")
    ps_out  = os.path.join(run_dir, "per_subject.csv")

    print("\n=== Finalize ===")
    print("run_dir:", run_dir)
    assert os.path.exists(run_dir), "Run dir does not exist"

    # Copy splits (optional but nice to keep consistent)
    dst_splits = os.path.join(run_dir, "splits_loso.json")
    if not os.path.exists(dst_splits):
        shutil.copy2(SPLITS_JSON, dst_splits)
        print("copied splits_loso.json ✅")
    else:
        print("splits_loso.json already exists ✅")

    # Finalize per_window
    if os.path.exists(pw_part):
        dfw = pd.read_csv(pw_part)
        # light de-dup if resume ever appended duplicates
        key_cols = [c for c in ["dataset","fold","subject_id","window_index"] if c in dfw.columns]
        if len(key_cols) >= 2:
            dfw = dfw.drop_duplicates(subset=key_cols, keep="last")
        _safe_write_csv(dfw, pw_out)
        print("wrote per_window.csv ✅ | rows:", len(dfw))
    elif os.path.exists(pw_out):
        dfw = pd.read_csv(pw_out)
        print("per_window.csv already exists ✅ | rows:", len(dfw))
    else:
        raise FileNotFoundError("Neither per_window_partial.csv nor per_window.csv exists")

    # Finalize per_subject
    if os.path.exists(ps_part):
        dfs = pd.read_csv(ps_part)
        key_cols = [c for c in ["dataset","fold","subject_id"] if c in dfs.columns]
        if len(key_cols) >= 2:
            dfs = dfs.drop_duplicates(subset=key_cols, keep="last")
        _safe_write_csv(dfs, ps_out)
        print("wrote per_subject.csv ✅ | rows:", len(dfs))
    elif os.path.exists(ps_out):
        dfs = pd.read_csv(ps_out)
        print("per_subject.csv already exists ✅ | rows:", len(dfs))
    else:
        raise FileNotFoundError("Neither per_subject_partial.csv nor per_subject.csv exists")

finalize_partials(PROPOSED_WESAD_DIR)
finalize_partials(PROPOSED_DALIA_DIR)

print("\n✅ Done. Proposed det-val runs are now FINAL.")



=== Finalize ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval
splits_loso.json already exists ✅
wrote per_window.csv ✅ | rows: 1797
wrote per_subject.csv ✅ | rows: 10

=== Finalize ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval
splits_loso.json already exists ✅
wrote per_window.csv ✅ | rows: 3883
wrote per_subject.csv ✅ | rows: 14

✅ Done. Proposed det-val runs are now FINAL.


In [20]:
# =========================
# RUN: Proposed v6 det-val — PPG-only ablation
# =========================

# --- required sanity ---
required = ["splits", "device", "train_one_fold_v6"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"Missing: {missing}. Re-run your setup/model cells first.")

# --- grab arrays from memory (accept either *_X_fusion or *_X) ---
wesad_XF, _ = _get_first_existing_global(["wesad_X_fusion", "wesad_X"])
dalia_XF, _ = _get_first_existing_global(["dalia_X_fusion", "dalia_X"])

wesad_sqi, _ = _get_first_existing_global(["wesad_sqi01", "wesad_sqi_raw"])
dalia_sqi, _ = _get_first_existing_global(["dalia_sqi01", "dalia_sqi_raw"])

wesad_motion, _ = _get_first_existing_global(["wesad_motion"])
dalia_motion, _ = _get_first_existing_global(["dalia_motion"])

wesad_rr, _ = _get_first_existing_global(["wesad_rr"])
wesad_sid, _ = _get_first_existing_global(["wesad_sid"])

dalia_rr, _ = _get_first_existing_global(["dalia_rr"])
dalia_sid, _ = _get_first_existing_global(["dalia_sid"])

METHOD = "proposed_v6_ablation_ppg_only"

# WESAD
run_dir_w = run_loso_proposed_ablation_with_ckpt(
    dataset_name="WESAD",
    X_fusion=wesad_XF, y=wesad_rr, sid=wesad_sid,
    sqi01=wesad_sqi, motion=wesad_motion,
    splits=splits, device=device,
    train_one_fold_v6_fn=train_one_fold_v6,
    variant="ppg_only",
    method_name=METHOD,
    tag="ckpt_detval",
    resume=True
)

# DaLiA
run_dir_d = run_loso_proposed_ablation_with_ckpt(
    dataset_name="PPG-DaLiA",
    X_fusion=dalia_XF, y=dalia_rr, sid=dalia_sid,
    sqi01=dalia_sqi, motion=dalia_motion,
    splits=splits, device=device,
    train_one_fold_v6_fn=train_one_fold_v6,
    variant="ppg_only",
    method_name=METHOD,
    tag="ckpt_detval",
    resume=True
)

print("\nPPG-only run dirs:")
print("WESAD :", run_dir_w)
print("DaLiA :", run_dir_d)



=== proposed_v6_ablation_ppg_only | WESAD | variant=ppg_only ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_ablation_ppg_only\WESAD\20260128_224919_ckpt_detval
folds: 10 | already done: [1, 2, 3, 4, 5, 6, 7, 8, 9]
settings: epochs=60, steps/epoch=120, batch=64, lr=0.0005, wd=0.0001, patience=10
[SKIP] fold 1/10 already saved
[SKIP] fold 2/10 already saved
[SKIP] fold 3/10 already saved
[SKIP] fold 4/10 already saved
[SKIP] fold 5/10 already saved
[SKIP] fold 6/10 already saved
[SKIP] fold 7/10 already saved
[SKIP] fold 8/10 already saved
[SKIP] fold 9/10 already saved

[proposed_v6_ablation_ppg_only] WESAD fold 10/10 | test=11 | val=2 | train_n=8
  epoch 1/60 | train_loss=3.0290 | val_mae=3.953 (val_subject=2)
  epoch 2/60 | train_loss=2.0106 | val_mae=5.052 (val_subject=2)
  epoch 3/60 | train_loss=1.8266 | val_mae=4.319 (val_subject=2)
  epoch 4/60 | train_loss=1.7275 | val_mae=5.424 (val_subject=2)
  epoch 5/60 | train_loss=1.6912 | val_mae=4.722 (val_subject=2)
  epoch 6/60 | tra

In [21]:
# =========================
# RUN: Proposed v6 det-val — IMU-only ablation
# =========================

METHOD = "proposed_v6_ablation_imu_only"

# WESAD
run_dir_w = run_loso_proposed_ablation_with_ckpt(
    dataset_name="WESAD",
    X_fusion=wesad_XF, y=wesad_rr, sid=wesad_sid,
    sqi01=wesad_sqi, motion=wesad_motion,
    splits=splits, device=device,
    train_one_fold_v6_fn=train_one_fold_v6,
    variant="imu_only",
    method_name=METHOD,
    tag="ckpt_detval",
    resume=True
)

# DaLiA
run_dir_d = run_loso_proposed_ablation_with_ckpt(
    dataset_name="PPG-DaLiA",
    X_fusion=dalia_XF, y=dalia_rr, sid=dalia_sid,
    sqi01=dalia_sqi, motion=dalia_motion,
    splits=splits, device=device,
    train_one_fold_v6_fn=train_one_fold_v6,
    variant="imu_only",
    method_name=METHOD,
    tag="ckpt_detval",
    resume=True
)

print("\nIMU-only run dirs:")
print("WESAD :", run_dir_w)
print("DaLiA :", run_dir_d)



=== proposed_v6_ablation_imu_only | WESAD | variant=imu_only ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_ablation_imu_only\WESAD\20260129_082345_ckpt_detval
folds: 10 | already done: []
settings: epochs=60, steps/epoch=120, batch=64, lr=0.0005, wd=0.0001, patience=10

[proposed_v6_ablation_imu_only] WESAD fold 1/10 | test=2 | val=3 | train_n=8
  epoch 1/60 | train_loss=3.3446 | val_mae=2.400 (val_subject=3)
  epoch 2/60 | train_loss=1.7097 | val_mae=2.603 (val_subject=3)
  epoch 3/60 | train_loss=1.3200 | val_mae=2.519 (val_subject=3)
  epoch 4/60 | train_loss=1.1553 | val_mae=2.516 (val_subject=3)
  epoch 5/60 | train_loss=0.9870 | val_mae=2.607 (val_subject=3)
  epoch 6/60 | train_loss=0.9275 | val_mae=2.805 (val_subject=3)
  epoch 7/60 | train_loss=0.8737 | val_mae=2.674 (val_subject=3)
  epoch 8/60 | train_loss=0.7943 | val_mae=2.694 (val_subject=3)
  epoch 9/60 | train_loss=0.7899 | val_mae=2.639 (val_subject=3)
  epoch 10/60 | train_loss=0.7788 | val_mae=2.734 (val_subject=3

In [26]:
# =========================
# B) SCHEMA + COVERAGE SIGNALS (auto)
# =========================
import pandas as pd

def peek_csv(path, n=3):
    df = pd.read_csv(path, nrows=n)
    return df

def print_schema(method_label, ds):
    rec = RUN_INDEX[(method_label, ds)]
    if not rec["ok"]:
        print(f"\n[{method_label} | {ds}] MISSING")
        return

    print(f"\n[{method_label} | {ds}]")
    print("per_window:", rec["per_window"])
    dfw = peek_csv(rec["per_window"], n=3)
    print("  per_window cols:", dfw.columns.tolist())
    print(dfw.head(2))

    print("per_subject:", rec["per_subject"])
    dfs = peek_csv(rec["per_subject"], n=3)
    print("  per_subject cols:", dfs.columns.tolist())
    print(dfs.head(2))

# 1) show schema for all methods/datasets
for (method_label, ds), rec in RUN_INDEX.items():
    print_schema(method_label, ds)

# 2) pick ONE "best" per_window to inspect signals (prefer proposed tuned detval, else any)
preferred = RUN_INDEX.get(("proposed_v6_tuned_detval", "WESAD"), None)
example = preferred["per_window"] if preferred and preferred["ok"] else None
if example is None:
    # fallback: first ok per_window
    for rec in RUN_INDEX.values():
        if rec["ok"]:
            example = rec["per_window"]
            break

print("\n=== COVERAGE SIGNAL CHECK ===")
print("Example per_window:", example)

df = pd.read_csv(example)  # full read ok (files ~few thousand rows)
cands = [
    "sqi01","sqi","sqi_raw","sqi_ppg",
    "motion","motion_score","acc_energy","imu_energy",
    "rr_ref","rr_true","y_true",
    "rr_pred","y_pred",
    "dataset","method","subject_id","fold","val_subject"
]
print("Found candidates:")
for c in cands:
    if c in df.columns:
        print("  ✅", c)

print("\nMotion-like describe:")
for c in ["motion","motion_score","acc_energy","imu_energy"]:
    if c in df.columns:
        print("\n", c)
        print(df[c].describe())



[kazemi_strong | WESAD]
per_window: C:\Users\yasmi\rr_runs\kazemi_strong\WESAD\20260126_195201\per_window.csv
  per_window cols: ['window_index', 'subject_id', 'rr_ref', 'rr_pred', 'is_valid', 'dataset']
   window_index  subject_id     rr_ref    rr_pred  is_valid dataset
0             0           2  16.153847  14.460279      True   WESAD
1             1           2  22.641510  16.558496      True   WESAD
per_subject: C:\Users\yasmi\rr_runs\kazemi_strong\WESAD\20260126_195201\per_subject.csv
  per_subject cols: ['MAE', 'RMSE', 'N', 'best_val_mae', 'val_subject', 'subject_id']
        MAE      RMSE    N  best_val_mae  val_subject  subject_id
0  5.016815  5.930215  186      3.245475           10           2
1  3.273165  4.024952  197      4.522651            2           3

[kazemi_strong | PPG-DaLiA]
per_window: C:\Users\yasmi\rr_runs\kazemi_strong\PPG-DaLiA\20260126_195202\per_window.csv
  per_window cols: ['window_index', 'subject_id', 'rr_ref', 'rr_pred', 'is_valid', 'dataset']
   win

###### import numpy as np

def _sig(a, n=2000, seed=0):
    """Tiny fingerprint of an array: shape + mean/std + a sampled checksum."""
    rng = np.random.default_rng(seed)
    a = np.asarray(a)
    flat = a.reshape(-1)
    if flat.size == 0:
        return {"shape": a.shape, "mean": None, "std": None, "chk": None}
    k = min(n, flat.size)
    idx = rng.choice(flat.size, size=k, replace=False)
    samp = flat[idx].astype(np.float64)
    return {
        "shape": tuple(a.shape),
        "mean": float(np.nanmean(samp)),
        "std":  float(np.nanstd(samp)),
        "chk":  float(np.nansum(np.abs(samp)))
    }

def compare_inputs(prefix, X_ppg, X_imu, X_fusion=None):
    print(f"\n=== {prefix} input fingerprints ===")
    s_ppg = _sig(X_ppg, seed=1)
    s_imu = _sig(X_imu, seed=2)
    print("PPG :", s_ppg)
    print("IMU :", s_imu)

    same_shape = (s_ppg["shape"] == s_imu["shape"])
    close_chk  = (abs(s_ppg["chk"] - s_imu["chk"]) / max(1e-9, abs(s_ppg["chk"])) < 1e-6)

    print("PPG vs IMU same shape?", same_shape)
    print("PPG vs IMU checksum extremely close?", close_chk, "(if True, suspicious)")

    if X_fusion is not None:
        s_fus = _sig(X_fusion, seed=3)
        print("FUS :", s_fus)
        print("Fusion same shape as PPG?", s_fus["shape"] == s_ppg["shape"])
        print("Fusion same shape as IMU?", s_fus["shape"] == s_imu["shape"])

# --- run it (only runs if vars exist) ---
if all(k in globals() for k in ["wesad_X_ppg","wesad_X_imu","wesad_X_fusion"]):
    compare_inputs("WESAD", wesad_X_ppg, wesad_X_imu, wesad_X_fusion)
elif all(k in globals() for k in ["wesad_X_ppg","wesad_X_imu"]):
    compare_inputs("WESAD", wesad_X_ppg, wesad_X_imu)
else:
    print("WESAD arrays not found in memory (need wesad_X_ppg and wesad_X_imu).")

if all(k in globals() for k in ["dalia_X_ppg","dalia_X_imu","dalia_X_fusion"]):
    compare_inputs("PPG-DaLiA", dalia_X_ppg, dalia_X_imu, dalia_X_fusion)
elif all(k in globals() for k in ["dalia_X_ppg","dalia_X_imu"]):
    compare_inputs("PPG-DaLiA", dalia_X_ppg, dalia_X_imu)
else:
    print("DaLiA arrays not found in memory (need dalia_X_ppg and dalia_X_imu).")
